# 03_Data Preparation

## 1. Objectifs de la préparation

Cette étape consiste à préparer les tables AdventureWorks pour les analyses exploratoires et la modélisation Power BI.

Les transformations suivront les principes suivants :

- Préserver les données sources.
- Ne pas supprimer de lignes sans justification.
- Conserver les valeurs manquantes lorsqu'aucune valeur fiable ne permet de les remplacer.
- Créer les variables calculées nécessaires aux analyses.
- Vérifier les types de données et la cohérence des tables préparées.
- Enregistrer les résultats dans `data/processed/`.

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

# Définir le chemin du projet
BASE_DIR = Path.cwd().parent
DATA_PATH = BASE_DIR / "data" / "raw" / "AdventureWorks_Database.xlsx"

# Charger toutes les feuilles Excel
donnees = pd.read_excel(DATA_PATH, sheet_name=None)

# Affecter les feuilles aux différentes tables
dim_date = donnees["Sheet2"]
dim_customer = donnees["Sheet3"]
dim_product = donnees["Sheet4"]
fact_sales = donnees["Sheet5"]
dim_territory = donnees["Sheet6"]

print("Chargement terminé.")

Chargement terminé.


In [11]:
# Vérifier les dimensions des cinq tables
tables = {
    "DimDate": dim_date,
    "DimCustomer": dim_customer,
    "DimProduct": dim_product,
    "FactInternetSales": fact_sales,
    "DimSalesTerritory": dim_territory
}

for nom, table in tables.items():
    print(f"{nom}: {table.shape[0]} lignes et {table.shape[1]} colonnes")

DimDate: 1461 lignes et 16 colonnes
DimCustomer: 18484 lignes et 17 colonnes
DimProduct: 606 lignes et 13 colonnes
FactInternetSales: 58189 lignes et 13 colonnes
DimSalesTerritory: 11 lignes et 5 colonnes


In [19]:
# Récupérer le coût standard unitaire de chaque produit
fact_sales = fact_sales.merge(
    dim_product[["ProductKey", "StandardCost"]],
    on="ProductKey",
    how="left"
)

# Calculer le coût total estimé de chaque ligne de vente
fact_sales["CalculatedTotalCost"] = (
    fact_sales["OrderQuantity"] * fact_sales["StandardCost"]
)

print("Colonne CalculatedTotalCost créée.")

Colonne CalculatedTotalCost créée.


In [23]:
# Vérifier les nouvelles colonnes et quelques exemples
display(
    fact_sales[
        [
            "ProductKey",
            "OrderQuantity",
            "StandardCost",
            "TotalProductCost",
            "CalculatedTotalCost"
        ]
    ].head(10)
)

# Vérifier les valeurs manquantes dans le nouveau calcul
print(
    "Valeurs manquantes dans CalculatedTotalCost :",
    fact_sales["CalculatedTotalCost"].isna().sum()
)

,ProductKey,OrderQuantity,StandardCost,TotalProductCost,CalculatedTotalCost
0,310,2,2171.2942,2171.2942,4342.5884
1,346,2,1912.1544,1912.1544,3824.3088
2,346,2,1912.1544,1912.1544,3824.3088
3,336,2,413.1463,413.1463,826.2926
4,346,2,1912.1544,1912.1544,3824.3088
5,311,4,2171.2942,2171.2942,8685.1768
6,310,4,2171.2942,2171.2942,8685.1768
7,351,4,1898.0944,1898.0944,7592.3776
8,344,4,1912.1544,1912.1544,7648.6176
9,312,1,2171.2942,2171.2942,2171.2942


Valeurs manquantes dans CalculatedTotalCost : 0


In [27]:
# Vérifier l'intégrité de la table après la jointure
print("Nombre de lignes :", len(fact_sales))

print(
    "Nombre de doublons sur les identifiants de vente :",
    fact_sales.duplicated(
        subset=["SalesOrderNumber", "SalesOrderLineNumber"]
    ).sum()
)

print(
    "Nombre de valeurs manquantes dans StandardCost :",
    fact_sales["StandardCost"].isna().sum()
)

print(
    "Nombre de valeurs manquantes dans CalculatedTotalCost :",
    fact_sales["CalculatedTotalCost"].isna().sum()
)

Nombre de lignes : 58189
Nombre de doublons sur les identifiants de vente : 0
Nombre de valeurs manquantes dans StandardCost : 0
Nombre de valeurs manquantes dans CalculatedTotalCost : 0


In [31]:
# Examiner les types de données de la table de ventes
display(
    fact_sales.dtypes.to_frame(name="Type de données")
)

,Type de données
ProductKey,int64
OrderDate,datetime64[ns]
ShipDate,datetime64[ns]
CustomerKey,int64
PromotionKey,int64
SalesTerritoryKey,int64
SalesOrderNumber,object
SalesOrderLineNumber,int64
OrderQuantity,int64
UnitPrice,float64


In [35]:
# Actualiser le dictionnaire des tables
tables["FactInternetSales"] = fact_sales

In [39]:
# Examiner les types de données de toutes les tables
for nom, table in tables.items():
    print(f"\n--- {nom} ---")
    display(table.dtypes.to_frame(name="Type de données"))


--- DimDate ---


,Type de données
Date,datetime64[ns]
DateKey,int64
Year,int64
Quarter,object
MonthNum,int64
Month,object
FiscalYear,object
FiscalQuarter,object
FiscalMonthNum,int64
FiscalMonth,object



--- DimCustomer ---


,Type de données
CustomerKey,int64
FirstName,object
LastName,object
FullName,object
BirthDate,datetime64[ns]
MaritalStatus,object
Gender,object
YearlyIncome,int64
TotalChildren,int64
NumberChildrenAtHome,int64



--- DimProduct ---


,Type de données
ProductKey,int64
ProductName,object
SubCategory,object
Category,object
StandardCost,float64
Color,object
ListPrice,float64
DaysToManufacture,int64
ProductLine,object
ModelName,object



--- FactInternetSales ---


,Type de données
ProductKey,int64
OrderDate,datetime64[ns]
ShipDate,datetime64[ns]
CustomerKey,int64
PromotionKey,int64
SalesTerritoryKey,int64
SalesOrderNumber,object
SalesOrderLineNumber,int64
OrderQuantity,int64
UnitPrice,float64



--- DimSalesTerritory ---


,Type de données
SalesTerritoryKey,int64
Region,object
Country,object
Group,object
RegionImage,object


In [43]:
# Examiner les colonnes de coût standard
colonnes_cout = [
    "StandardCost_x",
    "StandardCost_y",
    "StandardCost"
]

display(
    fact_sales[colonnes_cout].describe().round(4)
)

# Comparer les valeurs entre les différentes colonnes
print("Comparaison des colonnes :")

print(
    "StandardCost_x = StandardCost_y :",
    fact_sales["StandardCost_x"].equals(
        fact_sales["StandardCost_y"]
    )
)

print(
    "StandardCost_x = StandardCost :",
    fact_sales["StandardCost_x"].equals(
        fact_sales["StandardCost"]
    )
)

print(
    "StandardCost_y = StandardCost :",
    fact_sales["StandardCost_y"].equals(
        fact_sales["StandardCost"]
    )
)

,StandardCost_x,StandardCost_y,StandardCost
count,58189.0000,58189.0000,58189.0000
mean,296.5392,296.5392,296.5392
std,560.1714,560.1714,560.1714
min,0.8565,0.8565,0.8565
25%,3.3623,3.3623,3.3623
50%,12.1924,12.1924,12.1924
75%,343.6496,343.6496,343.6496
max,2171.2942,2171.2942,2171.2942


Comparaison des colonnes :
StandardCost_x = StandardCost_y : True
StandardCost_x = StandardCost : True
StandardCost_y = StandardCost : True


In [51]:
# Afficher les colonnes actuellement présentes
print(fact_sales.columns.tolist())

['ProductKey', 'OrderDate', 'ShipDate', 'CustomerKey', 'PromotionKey', 'SalesTerritoryKey', 'SalesOrderNumber', 'SalesOrderLineNumber', 'OrderQuantity', 'UnitPrice', 'TotalProductCost', 'SalesAmount', 'TaxAmt', 'CalculatedTotalCost', 'StandardCost']


In [55]:
# Identifier les colonnes redondantes encore présentes
colonnes_a_supprimer = [
    col for col in ["StandardCost_x", "StandardCost_y"]
    if col in fact_sales.columns
]

# Supprimer uniquement les colonnes présentes
fact_sales = fact_sales.drop(columns=colonnes_a_supprimer)

print("Colonnes supprimées :", colonnes_a_supprimer)
print("Dimensions de FactInternetSales :", fact_sales.shape)

Colonnes supprimées : []
Dimensions de FactInternetSales : (58189, 15)


In [59]:
# Actualiser le dictionnaire des tables après nettoyage
tables["FactInternetSales"] = fact_sales

# Vérifier les dimensions des cinq tables
for nom, table in tables.items():
    print(f"{nom}: {table.shape[0]} lignes et {table.shape[1]} colonnes")

DimDate: 1461 lignes et 16 colonnes
DimCustomer: 18484 lignes et 17 colonnes
DimProduct: 606 lignes et 13 colonnes
FactInternetSales: 58189 lignes et 15 colonnes
DimSalesTerritory: 11 lignes et 5 colonnes


In [63]:
# Vérifier les types de données de la table de ventes
display(
    fact_sales.dtypes.to_frame(name="Type de données")
)

,Type de données
ProductKey,int64
OrderDate,datetime64[ns]
ShipDate,datetime64[ns]
CustomerKey,int64
PromotionKey,int64
SalesTerritoryKey,int64
SalesOrderNumber,object
SalesOrderLineNumber,int64
OrderQuantity,int64
UnitPrice,float64


In [67]:
# Identifier les colonnes contenant des dates dans chaque table
for nom, table in tables.items():
    colonnes_dates = [
        col for col in table.columns
        if "date" in col.lower()
    ]

    print(f"\n--- {nom} ---")

    for col in colonnes_dates:
        print(f"{col} : {table[col].dtype}")


--- DimDate ---
Date : datetime64[ns]
DateKey : int64

--- DimCustomer ---
BirthDate : datetime64[ns]
DateFirstPurchase : datetime64[ns]

--- DimProduct ---
StartDate : datetime64[ns]

--- FactInternetSales ---
OrderDate : datetime64[ns]
ShipDate : datetime64[ns]

--- DimSalesTerritory ---


In [71]:
# Recenser les valeurs manquantes dans chaque table
for nom, table in tables.items():

    valeurs_manquantes = table.isna().sum()
    colonnes_incompletes = valeurs_manquantes[
        valeurs_manquantes > 0
    ]

    print(f"\n--- {nom} ---")

    if colonnes_incompletes.empty:
        print("Aucune valeur manquante.")
    else:
        display(
            colonnes_incompletes.to_frame(
                name="Nombre de valeurs manquantes"
            )
        )


--- DimDate ---
Aucune valeur manquante.

--- DimCustomer ---


,Nombre de valeurs manquantes
DateFirstPurchase,566



--- DimProduct ---


,Nombre de valeurs manquantes
SubCategory,209
Category,209
StandardCost,211
Color,254
ListPrice,211
ProductLine,226
ModelName,209
ProductDescription,210



--- FactInternetSales ---
Aucune valeur manquante.

--- DimSalesTerritory ---


,Nombre de valeurs manquantes
Region,1
Country,1
Group,1


## Vérification des valeurs manquantes après préparation

Après les premières transformations, un nouveau contrôle des valeurs manquantes a été effectué sur les cinq tables.

Les résultats confirment que :

- `DimDate` ne contient aucune valeur manquante.
- `DimCustomer` conserve 566 dates de première acquisition manquantes.
- `DimProduct` conserve ses informations incomplètes sur plusieurs attributs.
- `DimSalesTerritory` présente trois valeurs manquantes sur une seule ligne.
- `FactInternetSales` ne présente aucune valeur manquante.

Aucune valeur manquante n'a été introduite par les transformations réalisées.

**Décision :** les valeurs manquantes identifiées pendant l'audit sont conservées afin de préserver l'information disponible dans les données sources. Aucune imputation arbitraire n'est appliquée.


In [77]:
# Définir le dossier de stockage des données préparées
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# Créer le dossier s'il n'existe pas
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Dossier de préparation :", PROCESSED_DIR)
print("Dossier disponible :", PROCESSED_DIR.exists())

Dossier de préparation : C:\Users\DELL\Desktop\Babou\Python\Projets_DATA\Checkpoint Fonctionnalités avancées de visualisation\AdventureWorks-Advanced-Analytics\data\processed
Dossier disponible : True


In [81]:
# Définir les noms des fichiers à exporter
fichiers = {
    "DimDate": dim_date,
    "DimCustomer": dim_customer,
    "DimProduct": dim_product,
    "FactInternetSales": fact_sales,
    "DimSalesTerritory": dim_territory
}

# Exporter chaque table au format CSV
for nom, table in fichiers.items():

    chemin_fichier = PROCESSED_DIR / f"{nom}.csv"

    table.to_csv(
        chemin_fichier,
        index=False,
        encoding="utf-8-sig",
        na_rep=""
    )

    print(f"{nom}.csv : exporté avec succès.")

DimDate.csv : exporté avec succès.
DimCustomer.csv : exporté avec succès.
DimProduct.csv : exporté avec succès.
FactInternetSales.csv : exporté avec succès.
DimSalesTerritory.csv : exporté avec succès.


In [85]:
# Vérifier la présence et la taille des fichiers exportés
for nom in fichiers:

    chemin_fichier = PROCESSED_DIR / f"{nom}.csv"

    print(
        f"{nom}.csv | "
        f"Existe : {chemin_fichier.exists()} | "
        f"Taille : {chemin_fichier.stat().st_size / 1024:.2f} Ko"
    )

DimDate.csv | Existe : True | Taille : 127.86 Ko
DimCustomer.csv | Existe : True | Taille : 2370.19 Ko
DimProduct.csv | Existe : True | Taille : 136.89 Ko
FactInternetSales.csv | Existe : True | Taille : 5257.92 Ko
DimSalesTerritory.csv | Existe : True | Taille : 1.21 Ko


In [89]:
# Définir les colonnes contenant des dates
dates_a_parser = {
    "DimDate": ["Date"],
    "DimCustomer": ["BirthDate", "DateFirstPurchase"],
    "DimProduct": ["StartDate"],
    "FactInternetSales": ["OrderDate", "ShipDate"],
    "DimSalesTerritory": []
}

# Recharger les fichiers CSV
tables_csv = {}

for nom in fichiers:

    chemin_fichier = PROCESSED_DIR / f"{nom}.csv"

    tables_csv[nom] = pd.read_csv(
        chemin_fichier,
        encoding="utf-8-sig",
        parse_dates=dates_a_parser[nom]
    )

print("Rechargement des cinq fichiers terminé.")

Rechargement des cinq fichiers terminé.


In [93]:
# Comparer les tables préparées avec les fichiers rechargés
for nom in fichiers:

    original = fichiers[nom]
    recharge = tables_csv[nom]

    print(f"\n--- {nom} ---")
    print("Dimensions originales :", original.shape)
    print("Dimensions rechargées :", recharge.shape)

    print(
        "Colonnes identiques :",
        original.columns.equals(recharge.columns)
    )

    print(
        "Nombre de valeurs manquantes identique :",
        original.isna().sum().equals(recharge.isna().sum())
    )


--- DimDate ---
Dimensions originales : (1461, 16)
Dimensions rechargées : (1461, 16)
Colonnes identiques : True
Nombre de valeurs manquantes identique : True

--- DimCustomer ---
Dimensions originales : (18484, 17)
Dimensions rechargées : (18484, 17)
Colonnes identiques : True
Nombre de valeurs manquantes identique : True

--- DimProduct ---
Dimensions originales : (606, 13)
Dimensions rechargées : (606, 13)
Colonnes identiques : True
Nombre de valeurs manquantes identique : True

--- FactInternetSales ---
Dimensions originales : (58189, 15)
Dimensions rechargées : (58189, 15)
Colonnes identiques : True
Nombre de valeurs manquantes identique : True

--- DimSalesTerritory ---
Dimensions originales : (11, 5)
Dimensions rechargées : (11, 5)
Colonnes identiques : True
Nombre de valeurs manquantes identique : True


In [97]:
from pandas.testing import assert_frame_equal

# Comparer les valeurs des tables originales et rechargées
for nom in fichiers:

    original = fichiers[nom]
    recharge = tables_csv[nom]

    print(f"\n--- {nom} ---")

    try:
        assert_frame_equal(
            original,
            recharge,
            check_dtype=False,
            check_exact=False,
            rtol=1e-9,
            atol=1e-12
        )

        print("Comparaison des valeurs : conforme.")

    except AssertionError as erreur:
        print("Différences détectées :")
        print(erreur)


--- DimDate ---
Comparaison des valeurs : conforme.

--- DimCustomer ---
Comparaison des valeurs : conforme.

--- DimProduct ---
Comparaison des valeurs : conforme.

--- FactInternetSales ---
Comparaison des valeurs : conforme.

--- DimSalesTerritory ---
Comparaison des valeurs : conforme.


# Conclusion – Data Preparation

## Bilan des transformations

La préparation des données AdventureWorks a permis de produire cinq tables exploitables pour les analyses exploratoires et la modélisation décisionnelle.

Les opérations suivantes ont été réalisées :

1. Chargement des cinq tables depuis le fichier Excel original.
2. Création de `CalculatedTotalCost`, calculé à partir de `OrderQuantity × StandardCost`.
3. Conservation de la colonne originale `TotalProductCost`.
4. Suppression des colonnes `StandardCost` redondantes après vérification de leur identité.
5. Vérification des types de données et des formats de dates.
6. Conservation des valeurs manquantes identifiées lors de l'audit.
7. Export des cinq tables au format CSV dans `data/processed/`.

## Résultats des exports

| Table | Nombre de lignes | Nombre de colonnes |
|---|---:|---:|
| DimDate | 1 461 | 16 |
| DimCustomer | 18 484 | 17 |
| DimProduct | 606 | 13 |
| FactInternetSales | 58 189 | 15 |
| DimSalesTerritory | 11 | 5 |
| **Total** | **78 751** | **66** |

## Validation des fichiers

Les fichiers CSV ont été rechargés avec Pandas et comparés aux DataFrames préparés.

Les contrôles confirment :

- Des dimensions identiques.
- Des noms et un ordre de colonnes identiques.
- Des nombres de valeurs manquantes identiques.
- Des valeurs conformes lors de la comparaison des données.

Aucune différence significative n'a été détectée lors de ces vérifications.

## Conclusion

Les cinq tables préparées sont disponibles dans `data/processed/`.

Les données sources originales ont été préservées dans `data/raw/`. Les transformations sont documentées et les fichiers exportés ont passé les contrôles de validation.

**Statut : préparation des données terminée et exports validés.**

**Prochaine étape : analyse exploratoire des données (EDA) afin d'identifier les tendances commerciales, les comportements clients et les performances des produits et territoires.**